# Trial Eligibility

**Assess number of patients in the aUC analytic cohort meeting KEYNOTE-361 ECOG and lab eligibility.**

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorUrothelial, merge_dataframes

## Import dfs

In [2]:
index_df = pd.read_csv('../outputs/pembro_carbo_index.csv')

In [3]:
index_df.shape

(3712, 4)

In [4]:
dtype_map = pd.read_csv('../outputs/pembro_carbo_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembro_carbo_features_df.csv', dtype = dtype_map)

In [5]:
features_df.shape

(3706, 162)

In [6]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [7]:
surv_pred_df.shape

(3138, 2)

In [8]:
main_df = pd.merge(features_df, index_df, on = 'PatientID', how = 'left')

In [9]:
main_df = pd.merge(main_df, surv_pred_df, on = 'PatientID', how = 'left')

In [10]:
main_df = main_df.query('adv_diagnosis_year <= 2021')

In [11]:
main_df.shape

(3138, 166)

In [12]:
main_df = main_df[['PatientID', 'LineName', 'StartDate', 'psurv_180_calibrated']]

In [13]:
main_df.head(3)

,PatientID,LineName,StartDate,psurv_180_calibrated
0,F5AAF96C85477,pembro,2021-07-08,0.622046
3,F412959B03189,carbo,2020-02-19,0.622046
4,F6E944C1709E6,pembro,2020-08-12,0.804389


## ECOG and Labs

In [14]:
# Initialize class 
processor = DataProcessorUrothelial()

In [15]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = main_df,
                                 index_date_column = 'StartDate',
                                 additional_loinc_mappings = {'neutrophil': ['26499-4', '751-8', '753-4']},
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-04-25 23:05:14,736 - INFO - Successfully read Lab.csv file with shape: (9373598, 17) and unique PatientIDs: 12700
2026-04-25 23:05:17,139 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (2709906, 18) and unique PatientIDs: 3118
2026-04-25 23:05:23,505 - INFO - Successfully processed Lab.csv file with final shape: (3138, 81) and unique PatientIDs: 3138


In [16]:
labs_df = labs_df[['PatientID', 'neutrophil', 'hemoglobin', 'platelet', 'creatinine', 'total_bilirubin', 'ast', 'alt', 'alp']]

In [17]:
labs_df.shape

(3138, 9)

In [18]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = main_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-04-25 23:05:23,624 - INFO - Successfully read ECOG.csv file with shape: (184794, 4) and unique PatientIDs: 9933
2026-04-25 23:05:23,662 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (57057, 5) and unique PatientIDs: 2594
2026-04-25 23:05:23,712 - INFO - Successfully processed ECOG.csv file with final shape: (3138, 3) and unique PatientIDs: 3138


In [19]:
ecog_df = ecog_df[['PatientID', 'ecog_index']]

In [20]:
ecog_df.shape

(3138, 2)

In [21]:
main_df = pd.merge(main_df, labs_df, on = 'PatientID', how = 'left')

In [22]:
main_df = pd.merge(main_df, ecog_df, on = 'PatientID', how = 'left')

In [23]:
main_df.shape

(3138, 13)

In [24]:
main_df.head(3)

,PatientID,LineName,StartDate,psurv_180_calibrated,neutrophil,hemoglobin,platelet,creatinine,total_bilirubin,ast,alt,alp,ecog_index
0,F5AAF96C85477,pembro,2021-07-08,0.622046,3.70,10.9,106.0,1.51,0.4,21.0,14.0,64.0,2
1,F412959B03189,carbo,2020-02-19,0.622046,9.32,8.6,318.0,0.70,0.3,11.0,11.0,104.0,1
2,F6E944C1709E6,pembro,2020-08-12,0.804389,5.48,9.7,401.0,1.20,0.5,16.0,10.0,59.0,1


## Risk stratification

In [25]:
with open('../outputs/crossover_survival_estimate.txt', 'r') as f:
    crossover_survival_estimate = float(f.read())

In [26]:
main_df['high_risk'] = np.where(main_df['psurv_180_calibrated'] < crossover_survival_estimate, 1, 0)

In [27]:
main_df.high_risk.value_counts()

high_risk
0    2273
1     865
Name: count, dtype: int64

## ECOG Eligibility

In [28]:
main_df['ecog_eligible'] = np.where((main_df['ecog_index'] <= 2), 1, 0)

In [29]:
main_df['ecog_ineligible'] = np.where(main_df['ecog_index'] > 2, 1, 0)

In [30]:
main_df['ecog_indeterminate'] = np.where(main_df['ecog_index'].isna(), 1, 0)

In [31]:
main_df.shape[0]

3138

In [32]:
main_df.ecog_eligible.value_counts()[1]+main_df.ecog_ineligible.value_counts()[1]+main_df.ecog_indeterminate.value_counts()[1]

np.int64(3138)

## Lab eligibility

In [33]:
main_df['lab_eligible'] = np.where(
    (main_df['neutrophil'] >= 1.5) &
    (main_df['platelet'] >= 100) &
    (main_df['hemoglobin'] >= 9) &
    (main_df['creatinine'] <= 1.8) & 
    (main_df['total_bilirubin'] <= 1.8) &
    (main_df['ast'] <= 100) &
    (main_df['alt'] <= 100) &
    (main_df['alp'] <= 300), 1, 0)

In [34]:
main_df['lab_ineligible'] = np.where(
    (main_df['neutrophil'] < 1.5) |
    (main_df['platelet'] < 100) |
    (main_df['hemoglobin'] < 9) |
    (main_df['creatinine'] > 1.8) | 
    (main_df['total_bilirubin'] > 1.8) |
    (main_df['ast'] > 100) |
    (main_df['alt'] > 100) |
    (main_df['alp'] > 300), 1, 0)

In [35]:
main_df['lab_indeterminate'] = np.where((main_df['lab_eligible'] == 0) & (main_df['lab_ineligible'] == 0), 1, 0)

In [36]:
main_df.lab_eligible.value_counts()[1]+main_df.lab_ineligible.value_counts()[1]+main_df.lab_indeterminate.value_counts()[1]

np.int64(3138)

## Overal eligibility

In [37]:
main_df['eligible'] = np.where((main_df['ecog_eligible'] == 1) & (main_df['lab_eligible'] == 1), 1, 0)

In [38]:
main_df['ineligible'] = np.where((main_df['ecog_ineligible'] == 1) | (main_df['lab_ineligible'] == 1), 1, 0)

In [39]:
main_df['indeterminate'] = np.where((main_df['eligible'] == 0) & (main_df['ineligible'] == 0), 1, 0)

In [40]:
print('Overall cohort')
print(f'Eligible: count {main_df.eligible.value_counts()[1]}, percentage {round(main_df.eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.ineligible.value_counts()[1]}, percentage {round(main_df.ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.indeterminate.value_counts()[1]}, percentage {round(main_df.indeterminate.value_counts(normalize = True)[1]*100, 2)}')

print('')
print('High-risk')
print(f'Eligible: count {main_df.query('high_risk == 1').eligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.query('high_risk == 1').ineligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.query('high_risk == 1').indeterminate.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').indeterminate.value_counts(normalize = True)[1]*100, 2)}')

print('')
print('Low-risk')
print(f'Eligible: count {main_df.query('high_risk == 0').eligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.query('high_risk == 0').ineligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.query('high_risk == 0').indeterminate.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').indeterminate.value_counts(normalize = True)[1]*100, 2)}')

Overall cohort
Eligible: count 1048, percentage 33.4
Ineligible: count 869, percentage 27.69
Indeterminate: count 1221, percentage 38.91

High-risk
Eligible: count 205, percentage 23.7
Ineligible: count 451, percentage 52.14
Indeterminate: count 209, percentage 24.16

Low-risk
Eligible: count 843, percentage 37.09
Ineligible: count 418, percentage 18.39
Indeterminate: count 1012, percentage 44.52
